In [ ]:
!pip install -q pandas numpy scikit-learn tensorflow nltk

In [ ]:
import os
import re
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving CEAS_08_cleaned.csv to CEAS_08_cleaned.csv
Saving machinewars_filtered_emails.json to machinewars_filtered_emails.json
Saving Nazario_cleaned.csv to Nazario_cleaned.csv
Saving Nigerian_Fraud_cleaned.csv to Nigerian_Fraud_cleaned.csv
Saving SpamAssasin_cleaned.csv to SpamAssasin_cleaned.csv


In [ ]:
import pandas as pd
import json
import re
from pathlib import Path


# ---------------------------------
# 1. Helpers
# ---------------------------------
def safe_str(x):
    if pd.isna(x):
        return ""
    s = str(x).strip()
    if s.lower() in {"null", "none", "nan"}:
        return ""
    return s


URL_PATTERN = re.compile(
    r"((?:https?://|www\.)[^\s<>\"'()]+)",
    re.IGNORECASE,
)


def extract_urls_from_text(text):
    text = safe_str(text)
    matches = URL_PATTERN.findall(text)

    seen = set()
    urls = []
    for url in matches:
        url = url.strip().rstrip('.,;:!?')
        if url and url not in seen:
            seen.add(url)
            urls.append(url)

    return urls


def normalize_url_value(value):
    if isinstance(value, list):
        value = " | ".join(safe_str(v) for v in value if safe_str(v))
    return safe_str(value)


def populate_url_column(df):
    if "url" not in df.columns:
        df["url"] = ""

    df["url"] = df["url"].apply(normalize_url_value)

    missing_mask = df["url"].eq("")
    if missing_mask.any():
        df.loc[missing_mask, "url"] = df.loc[missing_mask, "body"].apply(
            lambda body: " | ".join(extract_urls_from_text(body))
        )

    return df


def normalize_sender(sender):
    return safe_str(sender).lower()


def build_text_all_fields(row):
    sender = normalize_sender(row.get("sender", ""))
    subject = safe_str(row.get("subject", ""))
    body = safe_str(row.get("body", ""))
    url = safe_str(row.get("url", ""))

    return (
        f"[SENDER] {sender}\n"
        f"[SUBJECT] {subject}\n"
        f"[BODY] {body}\n"
        f"[URL] {url}"
    ).strip()


def build_text_all_fields_from_parts(sender, subject, body, url):
    return build_text_all_fields({
        "sender": sender,
        "subject": subject,
        "body": body,
        "url": url,
    })


# ---------------------------------
# 2. Label handling
# ---------------------------------
def normalize_machinewars_label(label, spam_as_phishing=False):
    """
    MachineWars:
      Phishing -> 1
      Legitimate/Valid/Ham -> 0
      Spam -> 1 if spam_as_phishing=True else excluded
    """
    if pd.isna(label):
        return None

    label = str(label).strip().lower()

    if label == "phishing":
        return 1

    if label in {"legitimate", "valid", "ham", "benign", "safe"}:
        return 0

    if label == "spam":
        return 1 if spam_as_phishing else None

    return None


def normalize_test_label(label):
    """
    For CEAS-style test sets.
    Spam is excluded here unless you explicitly want otherwise.
    """
    if pd.isna(label):
        return None

    label = str(label).strip().lower()

    if label == "phishing":
        return 1

    if label in {"legitimate", "valid", "ham", "benign", "safe"}:
        return 0

    return None


# ---------------------------------
# 3. MachineWars loader
# ---------------------------------
def load_machinewars(json_path_or_list, spam_as_phishing=False, dataset_name="machinewars"):
    """
    Expected MachineWars fields:
      sender, subject, body, type, url
    """
    if isinstance(json_path_or_list, (str, Path)):
        with open(json_path_or_list, "r", encoding="utf-8") as f:
            data = json.load(f)
    else:
        data = json_path_or_list

    df = pd.DataFrame(data).copy()

    if "type" in df.columns:
        df = df.rename(columns={"type": "label"})

    for col in ["subject", "body", "label"]:
        if col not in df.columns:
            df[col] = ""

    if "sender" not in df.columns:
        df["sender"] = ""

    df = populate_url_column(df)

    df["dataset"] = dataset_name
    df["label_raw"] = df["label"].astype(str).str.strip().str.lower()
    df["label_id"] = df["label_raw"].apply(
        lambda x: normalize_machinewars_label(x, spam_as_phishing=spam_as_phishing)
    )

    df = df[df["label_id"].notna()].copy()
    df["label_id"] = df["label_id"].astype(int)
    df["label"] = df["label_id"].map({0: "legitimate", 1: "phishing"})
    df["text"] = df.apply(build_text_all_fields, axis=1)

    df = df[[
        "dataset", "sender", "subject", "body", "url",
        "label_raw", "label", "label_id", "text"
    ]]

    return df


# ---------------------------------
# 4. CEAS-style test loader
# ---------------------------------
def load_ceas_style_csv(csv_path, dataset_name=None):
    """
    Assumes CEAS-style columns similar to:
      subject, body, label
    """
    csv_path = Path(csv_path)
    if dataset_name is None:
        dataset_name = csv_path.stem

    df = pd.read_csv(csv_path).copy()

    rename_map = {
        "Subject": "subject",
        "Body": "body",
        "Label": "label",
        "type": "label",
        "From": "sender",
        "Sender": "sender",
    }
    df = df.rename(columns=rename_map)

    for col in ["subject", "body", "label"]:
        if col not in df.columns:
            df[col] = ""

    if "sender" not in df.columns:
        df["sender"] = ""

    df = populate_url_column(df)

    df["dataset"] = dataset_name
    df["label_raw"] = df["label"].astype(str).str.strip().str.lower()
    df["label_id"] = df["label_raw"].apply(normalize_test_label)

    df = df[df["label_id"].notna()].copy()
    df["label_id"] = df["label_id"].astype(int)
    df["label"] = df["label_id"].map({0: "legitimate", 1: "phishing"})
    df["text"] = df.apply(build_text_all_fields, axis=1)

    df = df[[
        "dataset", "sender", "subject", "body", "url",
        "label_raw", "label", "label_id", "text"
    ]]

    return df


In [ ]:
machinewars_spam_as_phishing_df = load_machinewars(
    "machinewars_filtered_emails.json",
    spam_as_phishing=False,
    dataset_name="machinewars"
)
train_df, val_df = train_test_split(
    machinewars_spam_as_phishing_df,
    test_size=0.2,
    random_state=SEED,
    stratify=machinewars_spam_as_phishing_df["label_id"]
)

In [ ]:
from datasets import Dataset

train_ds = Dataset.from_pandas(
    train_df[["text", "label_id"]]
    .rename(columns={"label_id": "labels"})
    .reset_index(drop=True)
)

val_ds = Dataset.from_pandas(
    val_df[["text", "label_id"]]
    .rename(columns={"label_id": "labels"})
    .reset_index(drop=True)
)
test_paths = [
    "CEAS_08_cleaned.csv",
    "Nazario_cleaned.csv",
    "Nigerian_Fraud_cleaned.csv",
    "SpamAssasin_cleaned.csv",
]

test_dfs = [load_ceas_style_csv(p) for p in test_paths]

for i, df in enumerate(test_dfs, 1):
    print(f"\nTest dataset {i}:")
    print(df["label"].value_counts())
    print(df.head(2))


Test dataset 1:
label
phishing      21842
legitimate    17312
Name: count, dtype: int64
           dataset                            sender  \
0  CEAS_08_cleaned  Young Esposito <Young@iworld.de>   
1  CEAS_08_cleaned      Mok <ipline's1983@icable.ph>   

                     subject  \
0  Never agree to be a loser   
1     Befriend Jenna Jameson   

                                                body  \
0  Buck up, your troubles caused by small dimensi...   
1  \nUpgrade your sex and pleasures with these te...   

                         url label_raw     label  label_id  \
0      http://whitedone.com/  phishing  phishing         1   
1  http://www.brightmade.com  phishing  phishing         1   

                                                text  
0  [SENDER] young esposito <young@iworld.de>\n[SU...  
1  [SENDER] mok <ipline's1983@icable.ph>\n[SUBJEC...  

Test dataset 2:
label
phishing    1565
Name: count, dtype: int64
           dataset                                        

In [ ]:
def compute_binary_metrics(y_true, y_pred, y_prob):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        zero_division=0
    )
    acc = accuracy_score(y_true, y_pred)

    try:
        roc_auc = roc_auc_score(y_true, y_prob)
    except Exception:
        roc_auc = float("nan")

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
    }

In [ ]:
X_train = train_df["text"].astype(str).tolist()
y_train = train_df["label_id"].astype(int).values

X_val = val_df["text"].astype(str).tolist()
y_val = val_df["label_id"].astype(int).values

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

tfidf_vectorizer = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_val_tfidf = tfidf_vectorizer.transform(X_val)
# Base linear SVM
svm_base = LinearSVC(
    class_weight="balanced",
    random_state=SEED,
    max_iter=5000
)

# Calibrated wrapper to enable predict_proba
svm_model = CalibratedClassifierCV(
    estimator=svm_base,
    method="sigmoid",
    cv=3
)

svm_model.fit(X_train_tfidf, y_train)

print("SVM training complete.")

SVM training complete.


In [ ]:
def svm_predict_one(text):
    X = tfidf_vectorizer.transform([str(text)])
    prob = svm_model.predict_proba(X)[0, 1]
    pred = int(prob >= 0.5)
    return pred, float(prob)


def svm_batch_predict(texts):
    X = tfidf_vectorizer.transform([str(t) for t in texts])
    probs = svm_model.predict_proba(X)[:, 1]
    preds = (probs >= 0.5).astype(int)
    return preds, probs


def evaluate_svm(df_eval):
    y_true = df_eval["label_id"].astype(int).values
    y_pred, y_prob = svm_batch_predict(df_eval["text"].astype(str).tolist())
    return compute_binary_metrics(y_true, y_pred, y_prob)

In [ ]:
print("Validation metrics:")
print(evaluate_svm(val_df))

svm_rows = [{"dataset": "validation", **evaluate_svm(val_df)}]

for test_df in test_dfs:
    test_name = test_df["dataset"].iloc[0]
    metrics = evaluate_svm(test_df)
    svm_rows.append({"dataset": test_name, **metrics})

svm_results_df = pd.DataFrame(svm_rows)
svm_results_df

Validation metrics:
{'accuracy': 0.9940119760479041, 'precision': 0.9962852897473997, 'recall': 0.9918639053254438, 'f1': 0.994069681245367, 'roc_auc': np.float64(0.9998716828043751)}


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


,dataset,accuracy,precision,recall,f1,roc_auc
0,validation,0.994012,0.996285,0.991864,0.994070,0.999872
1,CEAS_08_cleaned,0.466466,0.977912,0.044593,0.085296,0.821050
2,Nazario_cleaned,0.936102,1.000000,0.936102,0.966997,NaN
3,Nigerian_Fraud_cleaned,0.849640,1.000000,0.849640,0.918708,NaN
4,SpamAssasin_cleaned,0.725770,0.962963,0.075669,0.140313,0.923488


Epoch 1/5
334/334 ━━━━━━━━━━━━━━━━━━━━ 129s 369ms/step - accuracy: 0.9424 - loss: 0.1615 - val_accuracy: 0.9749 - val_loss: 0.0804
Epoch 2/5
334/334 ━━━━━━━━━━━━━━━━━━━━ 146s 382ms/step - accuracy: 0.9898 - loss: 0.0329 - val_accuracy: 0.9805 - val_loss: 0.0847
Epoch 3/5
334/334 ━━━━━━━━━━━━━━━━━━━━ 124s 372ms/step - accuracy: 0.9960 - loss: 0.0158 - val_accuracy: 0.9734 - val_loss: 0.0968


Validation metrics:
{'accuracy': 0.9749251497005988, 'precision': 0.9665940450254176, 'recall': 0.9844674556213018, 'f1': 0.9754488823744961, 'roc_auc': np.float64(0.996421126949973)}


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


,dataset,accuracy,precision,recall,f1,roc_auc
0,validation,0.974925,0.966594,0.984467,0.975449,0.996421
1,CEAS_08_cleaned,0.734791,0.957515,0.548942,0.697823,0.897904
2,Nazario_cleaned,0.943131,1.000000,0.943131,0.970733,NaN
3,Nigerian_Fraud_cleaned,0.888355,1.000000,0.888355,0.940877,NaN
4,SpamAssasin_cleaned,0.793080,0.822500,0.383003,0.522637,0.828668


In [ ]:
import nltk
from nltk.corpus import wordnet

nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [ ]:
def benign_prefix_attack(text):
    prefix = (
        "Hello,\n\n"
        "This is a routine business communication regarding normal account activity. "
        "Please review the information below.\n\n"
    )
    return prefix + str(text)


def benign_suffix_attack(text):
    suffix = (
        "\n\nThank you for your attention."
        "\nThis message is part of a standard informational workflow."
        "\nBest regards."
    )
    return str(text) + suffix


def contradiction_attack(text):
    prefix = (
        "Important note: this is a legitimate and safe message. "
        "It does not request any unsafe action.\n\n"
    )
    return prefix + str(text)


def prefix_injection_attack(text):
    prefix = (
        "This is a normal and trustworthy business email. "
        "The message is legitimate, safe, and routine.\n\n"
    )
    return prefix + str(text)


PHISHING_KEYWORDS = {
    "verify", "verification", "account", "password", "login", "signin",
    "security", "alert", "urgent", "confirm", "suspend", "suspended",
    "click", "update", "reset", "limited", "immediately"
}

def keyword_deletion_attack(text, max_delete=5):
    words = str(text).split()
    new_words = []
    deleted = 0

    for w in words:
        clean = re.sub(r"^[^\w]+|[^\w]+$", "", w).lower()
        if clean in PHISHING_KEYWORDS and deleted < max_delete:
            deleted += 1
            continue
        new_words.append(w)

    return " ".join(new_words)


def get_synonyms(word):
    syns = set()
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            s = lemma.name().replace("_", " ").strip()
            if s and s.lower() != word.lower():
                syns.add(s)
    return list(syns)


def synonym_attack(text, replace_prob=0.12, max_replacements=8, seed=42):
    rng = random.Random(seed)
    words = str(text).split()
    new_words = []
    replacements = 0

    for w in words:
        clean = re.sub(r"^[^\w]+|[^\w]+$", "", w)

        if (
            replacements < max_replacements
            and len(clean) >= 4
            and clean.isalpha()
            and rng.random() < replace_prob
        ):
            syns = get_synonyms(clean)
            syns = [s for s in syns if s.isalpha() and len(s.split()) == 1]

            if syns:
                replacement = rng.choice(syns)
                if w.istitle():
                    replacement = replacement.title()
                new_words.append(replacement)
                replacements += 1
                continue

        new_words.append(w)

    return " ".join(new_words)

In [ ]:
predict_one = svm_predict_one
batch_predict = svm_batch_predict
ACTIVE_MODEL_NAME = "svm"
print("Active model:", ACTIVE_MODEL_NAME)

Active model: svm


Active model: lstm


In [ ]:
def apply_attack_to_fields(row, subject_attack_fn=None, body_attack_fn=None):
    sender = row.get("sender", "")
    subject = safe_str(row.get("subject", ""))
    body = safe_str(row.get("body", ""))
    url = safe_str(row.get("url", ""))

    if subject_attack_fn is not None:
        subject = subject_attack_fn(subject)

    if body_attack_fn is not None:
        body = body_attack_fn(body)

    return build_text_all_fields_from_parts(sender, subject, body, url)


def evaluate_attack_common(df_eval, attack_name, row_attack_fn):
    attacked_texts = [row_attack_fn(row) for _, row in df_eval.iterrows()]
    y_true = df_eval["label_id"].astype(int).values

    y_pred, y_prob = batch_predict(attacked_texts)

    return {
        "attack": attack_name,
        "n_samples": len(df_eval),
        **compute_binary_metrics(y_true, y_pred, y_prob)
    }


In [ ]:
attack_rows_val = []

attack_rows_val.append(evaluate_attack_common(val_df, "clean", lambda row: row["text"]))
attack_rows_val.append(evaluate_attack_common(val_df, "benign_prefix", lambda row: apply_attack_to_fields(row, body_attack_fn=benign_prefix_attack)))
attack_rows_val.append(evaluate_attack_common(val_df, "benign_suffix", lambda row: apply_attack_to_fields(row, body_attack_fn=benign_suffix_attack)))
attack_rows_val.append(evaluate_attack_common(val_df, "contradiction", lambda row: apply_attack_to_fields(row, body_attack_fn=contradiction_attack)))
attack_rows_val.append(
    evaluate_attack_common(
        val_df,
        "synonym_attack",
        lambda row: apply_attack_to_fields(
            row,
            subject_attack_fn=lambda x: synonym_attack(x, replace_prob=0.12, max_replacements=8, seed=42),
            body_attack_fn=lambda x: synonym_attack(x, replace_prob=0.12, max_replacements=8, seed=42),
        )
    )
)
attack_rows_val.append(
    evaluate_attack_common(
        val_df,
        "keyword_deletion",
        lambda row: apply_attack_to_fields(
            row,
            subject_attack_fn=lambda x: keyword_deletion_attack(x, max_delete=5),
            body_attack_fn=lambda x: keyword_deletion_attack(x, max_delete=5),
        )
    )
)
attack_rows_val.append(evaluate_attack_common(val_df, "prefix_injection", lambda row: apply_attack_to_fields(row, body_attack_fn=prefix_injection_attack)))

attack_results_val_df = pd.DataFrame(attack_rows_val).sort_values("f1", ascending=False)
attack_results_val_df


,attack,n_samples,accuracy,precision,recall,f1,roc_auc
4,synonym_attack,2672,0.994386,0.997766,0.991124,0.994434,0.999863
5,keyword_deletion,2672,0.994386,0.997766,0.991124,0.994434,0.999888
6,prefix_injection,2672,0.994012,0.993353,0.994822,0.994087,0.999871
0,clean,2672,0.994012,0.996285,0.991864,0.994070,0.999872
2,benign_suffix,2672,0.994012,0.996285,0.991864,0.994070,0.999873
3,contradiction,2672,0.993638,0.994078,0.993343,0.993711,0.999868
1,benign_prefix,2672,0.993263,0.991888,0.994822,0.993353,0.999873


In [ ]:
all_attack_tables = {}

for test_df in test_dfs:
    test_name = test_df["dataset"].iloc[0]

    rows = []
    rows.append(evaluate_attack_common(test_df, "clean", lambda row: row["text"]))
    rows.append(evaluate_attack_common(test_df, "benign_prefix", lambda row: apply_attack_to_fields(row, body_attack_fn=benign_prefix_attack)))
    rows.append(evaluate_attack_common(test_df, "benign_suffix", lambda row: apply_attack_to_fields(row, body_attack_fn=benign_suffix_attack)))
    rows.append(evaluate_attack_common(test_df, "contradiction", lambda row: apply_attack_to_fields(row, body_attack_fn=contradiction_attack)))
    rows.append(
        evaluate_attack_common(
            test_df,
            "synonym_attack",
            lambda row: apply_attack_to_fields(
                row,
                subject_attack_fn=lambda x: synonym_attack(x, replace_prob=0.12, max_replacements=8, seed=42),
                body_attack_fn=lambda x: synonym_attack(x, replace_prob=0.12, max_replacements=8, seed=42),
            )
        )
    )
    rows.append(
        evaluate_attack_common(
            test_df,
            "keyword_deletion",
            lambda row: apply_attack_to_fields(
                row,
                subject_attack_fn=lambda x: keyword_deletion_attack(x, max_delete=5),
                body_attack_fn=lambda x: keyword_deletion_attack(x, max_delete=5),
            )
        )
    )
    rows.append(evaluate_attack_common(test_df, "prefix_injection", lambda row: apply_attack_to_fields(row, body_attack_fn=prefix_injection_attack)))

    result_df = pd.DataFrame(rows).sort_values("f1", ascending=False)
    all_attack_tables[test_name] = result_df

    print(f"\n=== {ACTIVE_MODEL_NAME} | {test_name} ===")
    print(result_df)



=== svm | CEAS_08_cleaned ===
             attack  n_samples  accuracy  precision    recall        f1  \
1     benign_prefix      39154  0.581473   0.976253  0.255975  0.405600   
6  prefix_injection      39154  0.521965   0.980326  0.146003  0.254154   
3     contradiction      39154  0.492491   0.980029  0.092116  0.168403   
2     benign_suffix      39154  0.472825   0.981556  0.056039  0.106025   
0             clean      39154  0.466466   0.977912  0.044593  0.085296   
5  keyword_deletion      39154  0.465112   0.981779  0.041938  0.080439   
4    synonym_attack      39154  0.465010   0.977588  0.041938  0.080425   

    roc_auc  
1  0.860976  
6  0.857252  
3  0.854025  
2  0.848022  
0  0.821050  
5  0.823314  
4  0.814945  


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist


=== svm | Nazario_cleaned ===
             attack  n_samples  accuracy  precision    recall        f1  \
1     benign_prefix       1565  0.979553        1.0  0.979553  0.989671   
6  prefix_injection       1565  0.969968        1.0  0.969968  0.984755   
3     contradiction       1565  0.962300        1.0  0.962300  0.980788   
2     benign_suffix       1565  0.942492        1.0  0.942492  0.970395   
0             clean       1565  0.936102        1.0  0.936102  0.966997   
4    synonym_attack       1565  0.918211        1.0  0.918211  0.957362   
5  keyword_deletion       1565  0.891374        1.0  0.891374  0.942568   

   roc_auc  
1      NaN  
6      NaN  
3      NaN  
2      NaN  
0      NaN  
4      NaN  
5      NaN  


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist


=== svm | Nigerian_Fraud_cleaned ===
             attack  n_samples  accuracy  precision    recall        f1  \
1     benign_prefix       3332  0.911164        1.0  0.911164  0.953518   
6  prefix_injection       3332  0.895858        1.0  0.895858  0.945069   
3     contradiction       3332  0.883854        1.0  0.883854  0.938346   
2     benign_suffix       3332  0.857743        1.0  0.857743  0.923425   
0             clean       3332  0.849640        1.0  0.849640  0.918708   
4    synonym_attack       3332  0.831633        1.0  0.831633  0.908078   
5  keyword_deletion       3332  0.819028        1.0  0.819028  0.900511   

   roc_auc  
1      NaN  
6      NaN  
3      NaN  
2      NaN  
0      NaN  
4      NaN  
5      NaN  

=== svm | SpamAssasin_cleaned ===
             attack  n_samples  accuracy  precision    recall        f1  \
1     benign_prefix       5809  0.751076   0.917178  0.174040  0.292564   
6  prefix_injection       5809  0.739370   0.955357  0.124563  0.220391 

NameError: name 'logreg_results_df' is not defined

FileNotFoundError: [Errno 2] No such file or directory: '/content/results'

In [ ]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

In [ ]:
def benign_prefix_attack(text):
    prefix = (
        "Hello,\n\n"
        "This is a routine business communication regarding normal account activity. "
        "Please review the information below.\n\n"
    )
    return prefix + str(text)


def benign_suffix_attack(text):
    suffix = (
        "\n\nThank you for your attention."
        "\nThis message is part of a standard informational workflow."
        "\nBest regards."
    )
    return str(text) + suffix


def contradiction_attack(text):
    prefix = (
        "Important note: this is a legitimate and safe message. "
        "It does not request any unsafe action.\n\n"
    )
    return prefix + str(text)


def training_context_attack(text):
    prefix = (
        "Example of a safe enterprise email used for routine staff communication:\n\n"
    )
    return prefix + str(text)


def noise_injection_attack(text):
    noise = (
        "\n\nReference ID: A7K2-91LM"
        "\nDepartment: operations"
        "\nCase Type: routine review"
    )
    return str(text) + noise

In [ ]:
import re

STOPWORDS_ATTACK = {
    "the", "a", "an", "and", "or", "to", "of", "in", "for", "on", "at",
    "is", "are", "this", "that", "with", "from", "by", "as", "it", "be",
    "was", "were", "subject", "body", "sender", "url", "dear", "hello", "hi", "regards",
    "thanks", "thank", "best"
}

PROTECTED_TOKENS = {"[SENDER]", "[SUBJECT]", "[BODY]", "[URL]"}


def basic_tokenize_with_indices(text):
    """
    Splits text into whitespace-separated tokens and keeps positions.
    """
    tokens = str(text).split()
    return tokens


def is_deletable_token(tok):
    if tok in PROTECTED_TOKENS:
        return False

    clean = re.sub(r"^[^\w]+|[^\w]+$", "", tok).lower()
    if len(clean) < 3:
        return False
    if clean in STOPWORDS_ATTACK:
        return False
    if not any(ch.isalpha() for ch in clean):
        return False
    return True


def delete_token_at_index(tokens, idx):
    return " ".join(tokens[:idx] + tokens[idx+1:])


def greedy_delete_attack_blackbox(text, max_delete_steps=5, candidate_cap=15):
    """
    Black-box deletion attack:
    At each step, test candidate single-token deletions and keep the one that
    minimizes phishing probability.
    """
    current_text = str(text)

    for _ in range(max_delete_steps):
        tokens = basic_tokenize_with_indices(current_text)

        candidate_indices = [i for i, tok in enumerate(tokens) if is_deletable_token(tok)]

        if not candidate_indices:
            break

        candidate_indices = candidate_indices[:candidate_cap]
        candidate_texts = [delete_token_at_index(tokens, i) for i in candidate_indices]

        _, candidate_probs = batch_predict(candidate_texts)
        best_idx = int(np.argmin(candidate_probs))
        best_text = candidate_texts[best_idx]

        if best_text == current_text:
            break

        current_text = best_text

    return current_text


In [ ]:
def greedy_add_attack_blackbox(text, add_steps=3):
    """
    Greedily applies the benign addition that lowers phishing probability the most.
    """
    current_text = str(text)

    addition_fns = [
        ("benign_prefix", benign_prefix_attack),
        ("benign_suffix", benign_suffix_attack),
        ("contradiction", contradiction_attack),
        ("training_context", training_context_attack),
        ("noise_injection", noise_injection_attack),
    ]

    history = []

    for step in range(1, add_steps + 1):
        candidate_names = []
        candidate_texts = []

        for attack_name, attack_fn in addition_fns:
            candidate_names.append(attack_name)
            candidate_texts.append(attack_fn(current_text))

        candidate_preds, candidate_probs = batch_predict(candidate_texts)

        best_idx = int(np.argmin(candidate_probs))
        current_text = candidate_texts[best_idx]

        history.append({
            "step": step,
            "attack_name": candidate_names[best_idx],
            "pred": int(candidate_preds[best_idx]),
            "phishing_prob": float(candidate_probs[best_idx]),
        })

    return current_text, history


def greedy_delete_subject_body_blackbox(row, max_delete_steps=3, candidate_cap=5):
    sender = row.get("sender", "")
    subject = safe_str(row.get("subject", ""))
    body = safe_str(row.get("body", ""))
    url = safe_str(row.get("url", ""))

    current_subject = subject
    current_body = body

    for _ in range(max_delete_steps):
        subject_tokens = basic_tokenize_with_indices(current_subject)
        body_tokens = basic_tokenize_with_indices(current_body)

        subject_indices = [
            i for i, tok in enumerate(subject_tokens)
            if is_deletable_token(tok)
        ][:candidate_cap]

        remaining_cap = candidate_cap - len(subject_indices)

        body_indices = [
            i for i, tok in enumerate(body_tokens)
            if is_deletable_token(tok)
        ][:max(0, remaining_cap)]

        candidate_texts = []
        candidate_states = []

        for i in subject_indices:
            new_subject = delete_token_at_index(subject_tokens, i)
            candidate_texts.append(
                build_text_all_fields_from_parts(sender, new_subject, current_body, url)
            )
            candidate_states.append((new_subject, current_body))

        for i in body_indices:
            new_body = delete_token_at_index(body_tokens, i)
            candidate_texts.append(
                build_text_all_fields_from_parts(sender, current_subject, new_body, url)
            )
            candidate_states.append((current_subject, new_body))

        if not candidate_texts:
            break

        _, candidate_probs = batch_predict(candidate_texts)
        best_idx = int(np.argmin(candidate_probs))

        best_subject, best_body = candidate_states[best_idx]

        if best_subject == current_subject and best_body == current_body:
            break

        current_subject = best_subject
        current_body = best_body

    return build_text_all_fields_from_parts(sender, current_subject, current_body, url)

def greedy_add_to_body_blackbox(row, add_steps=3):
    sender = row.get("sender", "")
    subject = safe_str(row.get("subject", ""))
    body = safe_str(row.get("body", ""))
    url = safe_str(row.get("url", ""))

    attacked_body, history = greedy_add_attack_blackbox(body, add_steps=add_steps)
    attacked_text = build_text_all_fields_from_parts(sender, subject, attacked_body, url)
    return attacked_text, history


In [ ]:
def add_only_attack(row, add_steps=3):
    attacked_text, _ = greedy_add_to_body_blackbox(row, add_steps=add_steps)
    return attacked_text


def delete_only_attack(row, delete_steps=5):
    attacked_text = greedy_delete_subject_body_blackbox(row, max_delete_steps=delete_steps)
    return attacked_text


def hybrid_add_then_delete_attack_blackbox(row, add_steps=3, delete_steps=5):
    """
    First greedy additions to body, then greedy deletions on subject/body.
    """
    sender = row.get("sender", "")
    subject = safe_str(row.get("subject", ""))
    body = safe_str(row.get("body", ""))
    url = safe_str(row.get("url", ""))

    current_body, _ = greedy_add_attack_blackbox(body, add_steps=add_steps)
    temp_row = {
        "sender": sender,
        "subject": subject,
        "body": current_body,
        "url": url,
    }
    current_text = greedy_delete_subject_body_blackbox(temp_row, max_delete_steps=delete_steps)
    return current_text


In [ ]:
def evaluate_attack_detailed(df_eval, attack_name, row_attack_fn, show_progress=True):
    y_true = df_eval["label_id"].astype(int).tolist()
    texts = df_eval["text"].astype(str).tolist()

    orig_pred, orig_prob = batch_predict(texts)

    attacked_texts = []
    if show_progress:
        iterator = tqdm(df_eval.iterrows(), total=len(df_eval), desc=f"{ACTIVE_MODEL_NAME} | {attack_name}")
    else:
        iterator = df_eval.iterrows()

    for _, row in iterator:
        attacked_texts.append(row_attack_fn(row))

    new_pred, new_prob = batch_predict(attacked_texts)

    metrics = compute_binary_metrics(y_true, new_pred, new_prob)

    details_df = pd.DataFrame({
        "text": texts,
        "label_id": y_true,
        "orig_pred": orig_pred,
        "orig_prob": orig_prob,
        "attacked_text": attacked_texts,
        "new_pred": new_pred,
        "new_prob": new_prob,
    })

    details_df["flipped"] = details_df["orig_pred"] != details_df["new_pred"]
    details_df["prob_drop"] = details_df["orig_prob"] - details_df["new_prob"]

    summary = {
        "attack": attack_name,
        "n_samples": len(df_eval),
        "flip_rate": float(details_df["flipped"].mean()),
        "avg_prob_drop": float(details_df["prob_drop"].mean()),
        **metrics,
    }

    return summary, details_df


In [ ]:
def evaluate_evasion_on_phishing(df_eval, attack_name, row_attack_fn, show_progress=True):
    """
    Restrict evaluation to:
    - true phishing samples
    - originally correct phishing predictions

    Reports attack success rate (ASR).
    """
    df_local = df_eval.copy()
    df_local = df_local[df_local["label_id"].astype(int) == 1].copy()

    if len(df_local) == 0:
        return {
            "attack": attack_name,
            "n_true_phishing": 0,
            "n_orig_correct_phishing": 0,
            "n_flipped": 0,
            "attack_success_rate": float("nan"),
            "robust_recall_on_orig_correct_phishing": float("nan"),
            "avg_prob_drop": float("nan"),
        }, pd.DataFrame()

    texts = df_local["text"].astype(str).tolist()
    orig_pred, orig_prob = batch_predict(texts)

    df_local["orig_pred"] = orig_pred
    df_local["orig_prob"] = orig_prob

    df_attack = df_local[df_local["orig_pred"] == 1].copy()

    if len(df_attack) == 0:
        return {
            "attack": attack_name,
            "n_true_phishing": len(df_local),
            "n_orig_correct_phishing": 0,
            "n_flipped": 0,
            "attack_success_rate": float("nan"),
            "robust_recall_on_orig_correct_phishing": float("nan"),
            "avg_prob_drop": float("nan"),
        }, pd.DataFrame()

    attacked_texts = []
    if show_progress:
        iterator = tqdm(df_attack.iterrows(), total=len(df_attack), desc=f"{ACTIVE_MODEL_NAME} | {attack_name} phishing")
    else:
        iterator = df_attack.iterrows()

    for _, row in iterator:
        attacked_texts.append(row_attack_fn(row))

    new_pred, new_prob = batch_predict(attacked_texts)

    details_df = pd.DataFrame({
        "text": df_attack["text"].astype(str).tolist(),
        "label_id": df_attack["label_id"].astype(int).tolist(),
        "orig_pred": df_attack["orig_pred"].tolist(),
        "orig_prob": df_attack["orig_prob"].tolist(),
        "attacked_text": attacked_texts,
        "new_pred": new_pred,
        "new_prob": new_prob,
    })

    details_df["flipped"] = details_df["new_pred"] != 1
    details_df["prob_drop"] = details_df["orig_prob"] - details_df["new_prob"]

    n_orig_correct = len(details_df)
    n_flipped = int(details_df["flipped"].sum())
    asr = n_flipped / n_orig_correct
    robust_recall = 1.0 - asr

    summary = {
        "attack": attack_name,
        "n_true_phishing": len(df_local),
        "n_orig_correct_phishing": n_orig_correct,
        "n_flipped": n_flipped,
        "attack_success_rate": asr,
        "robust_recall_on_orig_correct_phishing": robust_recall,
        "avg_prob_drop": float(details_df["prob_drop"].mean()),
    }

    return summary, details_df


In [ ]:
val_attack_summaries = []
val_attack_details = {}

attack_specs = [
    ("add_only_add3", lambda row: add_only_attack(row, add_steps=3)),
    ("delete_only_del5", lambda row: delete_only_attack(row, delete_steps=5)),
    ("hybrid_add3_delete5", lambda row: hybrid_add_then_delete_attack_blackbox(row, add_steps=3, delete_steps=5)),
]


In [ ]:
for attack_name, attack_fn in attack_specs:
    summary, details = evaluate_attack_detailed(
        val_df,
        attack_name,
        attack_fn,
        show_progress=True
    )
    val_attack_summaries.append(summary)
    val_attack_details[attack_name] = details

val_attack_results_df = pd.DataFrame(val_attack_summaries).sort_values("f1", ascending=False)
val_attack_results_df

svm | add_only_add3:   0%|          | 0/2672 [00:00<?, ?it/s]

svm | delete_only_del5:   0%|          | 0/2672 [00:00<?, ?it/s]

svm | hybrid_add3_delete5:   0%|          | 0/2672 [00:00<?, ?it/s]

,attack,n_samples,flip_rate,avg_prob_drop,accuracy,precision,recall,f1,roc_auc
0,add_only_add3,2672,0.000000,-0.000636,0.994012,0.996285,0.991864,0.994070,0.999869
2,hybrid_add3_delete5,2672,0.001871,0.003919,0.993638,0.997763,0.989645,0.993687,0.999840
1,delete_only_del5,2672,0.002246,0.004186,0.993263,0.997761,0.988905,0.993314,0.999841


In [ ]:
val_evasion_summaries = []
val_evasion_details = {}

for attack_name, attack_fn in attack_specs:
    summary, details = evaluate_evasion_on_phishing(
        val_df,
        attack_name,
        attack_fn,
        show_progress=True
    )
    val_evasion_summaries.append(summary)
    val_evasion_details[attack_name] = details

val_evasion_results_df = pd.DataFrame(val_evasion_summaries).sort_values("attack_success_rate", ascending=False)
val_evasion_results_df

svm | add_only_add3 phishing:   0%|          | 0/1341 [00:00<?, ?it/s]

svm | delete_only_del5 phishing:   0%|          | 0/1341 [00:00<?, ?it/s]

svm | hybrid_add3_delete5 phishing:   0%|          | 0/1341 [00:00<?, ?it/s]

,attack,n_true_phishing,n_orig_correct_phishing,n_flipped,attack_success_rate,robust_recall_on_orig_correct_phishing,avg_prob_drop
1,delete_only_del5,1352,1341,4,0.002983,0.997017,0.004598
2,hybrid_add3_delete5,1352,1341,3,0.002237,0.997763,0.005514
0,add_only_add3,1352,1341,0,0.000000,1.000000,0.000557


In [ ]:
all_test_attack_results = {}
all_test_evasion_results = {}

for test_df in test_dfs:
    dataset_name = test_df["dataset"].iloc[0]

    print(f"\n\n==============================")
    print(f"Starting dataset: {dataset_name}")
    print(f"Rows: {len(test_df)}")
    print(f"==============================")

    attack_summaries = []
    evasion_summaries = []

    for attack_name, attack_fn in attack_specs:
        print(f"\nRunning attack: {attack_name}")

        summary_attack, _ = evaluate_attack_detailed(
            test_df,
            attack_name,
            attack_fn,
            show_progress=True
        )
        attack_summaries.append({"dataset": dataset_name, **summary_attack})

        summary_evasion, _ = evaluate_evasion_on_phishing(
            test_df,
            attack_name,
            attack_fn,
            show_progress=True
        )
        evasion_summaries.append({"dataset": dataset_name, **summary_evasion})

        print("\nAttack metrics:")
        display(pd.DataFrame([{"dataset": dataset_name, **summary_attack}]))

        print("\nPhishing evasion metrics:")
        display(pd.DataFrame([{"dataset": dataset_name, **summary_evasion}]))

    dataset_attack_df = pd.DataFrame(attack_summaries)
    dataset_evasion_df = pd.DataFrame(evasion_summaries)

    all_test_attack_results[dataset_name] = dataset_attack_df
    all_test_evasion_results[dataset_name] = dataset_evasion_df

    print(f"\nFinished dataset: {dataset_name}")

    print("\nAll attack metrics for this dataset:")
    display(dataset_attack_df)

    print("\nAll phishing evasion metrics for this dataset:")
    display(dataset_evasion_df)



Starting dataset: CEAS_08_cleaned
Rows: 39154

Running attack: add_only_add3


svm | add_only_add3:   0%|          | 0/39154 [00:00<?, ?it/s]

svm | add_only_add3 phishing:   0%|          | 0/974 [00:00<?, ?it/s]


Attack metrics:


,dataset,attack,n_samples,flip_rate,avg_prob_drop,accuracy,precision,recall,f1,roc_auc
0,CEAS_08_cleaned,add_only_add3,39154,0.005568,-0.036429,0.471318,0.979832,0.053383,0.10125,0.858077


...

,dataset,attack,n_samples,flip_rate,avg_prob_drop,accuracy,precision,recall,f1,roc_auc
0,Nazario_cleaned,add_only_add3,1565,0.005751,0.004789,0.941853,1.0,0.941853,0.970056,NaN



Phishing evasion metrics:


,dataset,attack,n_true_phishing,n_orig_correct_phishing,n_flipped,attack_success_rate,robust_recall_on_orig_correct_phishing,avg_prob_drop
0,Nazario_cleaned,add_only_add3,1565,1465,0,0.0,1.0,0.008587



Running attack: delete_only_del5


svm | delete_only_del5:   0%|          | 0/1565 [00:00<?, ?it/s]

In [ ]:
all_test_attack_results = {}
all_test_evasion_results = {}

for test_df in test_dfs:
    dataset_name = test_df["dataset"].iloc[0]

    if dataset_name == "CEAS_08_cleaned":
        print(f"\nSkipping dataset: {dataset_name}")
        continue

    print(f"\n\n==============================")
    print(f"Starting dataset: {dataset_name}")
    print(f"Rows: {len(test_df)}")
    print(f"==============================")

    attack_summaries = []
    evasion_summaries = []

    for attack_name, attack_fn in attack_specs:
        print(f"\nRunning attack: {attack_name}")

        summary_attack, _ = evaluate_attack_detailed(
            test_df,
            attack_name,
            attack_fn,
            show_progress=True
        )
        attack_summaries.append({"dataset": dataset_name, **summary_attack})

        summary_evasion, _ = evaluate_evasion_on_phishing(
            test_df,
            attack_name,
            attack_fn,
            show_progress=True
        )
        evasion_summaries.append({"dataset": dataset_name, **summary_evasion})

        print("\nAttack metrics:")
        display(pd.DataFrame([{"dataset": dataset_name, **summary_attack}]))

        print("\nPhishing evasion metrics:")
        display(pd.DataFrame([{"dataset": dataset_name, **summary_evasion}]))

    dataset_attack_df = pd.DataFrame(attack_summaries)
    dataset_evasion_df = pd.DataFrame(evasion_summaries)

    all_test_attack_results[dataset_name] = dataset_attack_df
    all_test_evasion_results[dataset_name] = dataset_evasion_df

    print(f"\nFinished dataset: {dataset_name}")

    print("\nAll attack metrics for this dataset:")
    display(dataset_attack_df)

    print("\nAll phishing evasion metrics for this dataset:")
    display(dataset_evasion_df)


Skipping dataset: CEAS_08_cleaned


Starting dataset: Nazario_cleaned
Rows: 1565

Running attack: add_only_add3


svm | add_only_add3:   0%|          | 0/1565 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


svm | add_only_add3 phishing:   0%|          | 0/1465 [00:00<?, ?it/s]


Attack metrics:


...

,dataset,attack,n_true_phishing,n_orig_correct_phishing,n_flipped,attack_success_rate,robust_recall_on_orig_correct_phishing,avg_prob_drop
0,SpamAssasin_cleaned,hybrid_add3_delete5,1718,130,15,0.115385,0.884615,0.050063



Finished dataset: SpamAssasin_cleaned

All attack metrics for this dataset:


,dataset,attack,n_samples,flip_rate,avg_prob_drop,accuracy,precision,recall,f1,roc_auc
0,SpamAssasin_cleaned,add_only_add3,5809,0.001205,-0.011609,0.726631,0.957746,0.079162,0.146237,0.895066
1,SpamAssasin_cleaned,delete_only_del5,5809,0.003615,0.006231,0.722500,0.964912,0.064028,0.120087,0.919338
2,SpamAssasin_cleaned,hybrid_add3_delete5,5809,0.002754,-0.005153,0.723360,0.966387,0.066938,0.125204,0.893163



All phishing evasion metrics for this dataset:


,dataset,attack,n_true_phishing,n_orig_correct_phishing,n_flipped,attack_success_rate,robust_recall_on_orig_correct_phishing,avg_prob_drop
0,SpamAssasin_cleaned,add_only_add3,1718,130,0,0.000000,1.000000,0.000616
1,SpamAssasin_cleaned,delete_only_del5,1718,130,20,0.153846,0.846154,0.061561
2,SpamAssasin_cleaned,hybrid_add3_delete5,1718,130,15,0.115385,0.884615,0.050063


In [ ]:
combined_attack_df = pd.concat(all_test_attack_results.values(), ignore_index=True)
combined_evasion_df = pd.concat(all_test_evasion_results.values(), ignore_index=True)

print("Combined attack results:")
print(combined_attack_df)

print("\nCombined evasion results:")
print(combined_evasion_df)

import os
os.makedirs("/content/results", exist_ok=True)

val_attack_results_df.to_csv(f"/content/results/{ACTIVE_MODEL_NAME}_val_add_delete_hybrid_metrics.csv", index=False)
val_evasion_results_df.to_csv(f"/content/results/{ACTIVE_MODEL_NAME}_val_add_delete_hybrid_evasion.csv", index=False)
combined_attack_df.to_csv(f"/content/results/{ACTIVE_MODEL_NAME}_test_add_delete_hybrid_metrics.csv", index=False)
combined_evasion_df.to_csv(f"/content/results/{ACTIVE_MODEL_NAME}_test_add_delete_hybrid_evasion.csv", index=False)

print("Saved.")